# 2. Data Security Design

SC-100 asks: **"Design a data security architecture that protects data at rest, in transit, and in use — across Azure, M365, and hybrid environments."**

## Setup

```bash
cd security-certs/sc-100/04-applications-and-data
uv sync
# Notebooks use the local .venv directly -- no global kernel to register.
# In VS Code: open the kernel picker (top-right) and select `.venv`.
# In classic Jupyter: uv run jupyter notebook notebooks/
```
Then pick the **`.venv` kernel** for this folder from the VS Code kernel picker (top-right).

## Data Classification Architecture

You can't protect data you haven't classified. Classification is the foundation of data security.

### Microsoft Purview Information Protection — classification hierarchy:

```
┌──────────────────────────────────────────────────────────────────────┐
│                    SENSITIVITY LABELS                               │
│                                                                      │
│  ┌─────────────┐  ┌─────────────┐  ┌──────────────┐  ┌──────────┐  │
│  │   PUBLIC     │  │  GENERAL    │  │ CONFIDENTIAL │  │ HIGHLY   │  │
│  │              │  │             │  │              │  │ CONFID.  │  │
│  │  No protect. │  │  No encrypt │  │  Encrypt     │  │  Encrypt │  │
│  │  No restrict │  │  Watermark  │  │  Restrict    │  │  No fwd  │  │
│  │              │  │             │  │  forwarding  │  │  No print│  │
│  │              │  │             │  │              │  │  No copy │  │
│  │  Examples:   │  │  Examples:  │  │  Sub-labels: │  │  Track   │  │
│  │  Marketing   │  │  Internal   │  │  • All Empl. │  │  Revoke  │  │
│  │  materials   │  │  memos      │  │  • Finance   │  │          │  │
│  │              │  │             │  │  • HR        │  │  Board   │  │
│  │              │  │             │  │  • Legal     │  │  minutes │  │
│  └─────────────┘  └─────────────┘  └──────────────┘  └──────────┘  │
│                                                                      │
│  Applied via:                                                        │
│  • Manual: User applies in Office apps / Outlook                     │
│  • Recommended: Purview suggests label based on content              │
│  • Auto-labeling: Service-side policy labels at rest (SharePoint)    │
│  • Default: Automatically applied to all new documents               │
└──────────────────────────────────────────────────────────────────────┘
```

### Classification methods:

| Method | How it works | Best for |
|--------|-------------|----------|
| Sensitive information types (SITs) | Regex + keyword + confidence | PII (SSN, credit card, passport) |
| Trainable classifiers | ML model trained on examples | Unstructured data (contracts, resumes) |
| Exact data match (EDM) | Hash match against your data | Exact customer records (employee IDs) |
| Fingerprinting | Document template matching | Standard forms (tax forms, applications) |

In [ ]:
import json

# ===================================================================
# ENCRYPTION STRATEGY DESIGN
# Different data states need different encryption approaches
# ===================================================================

ENCRYPTION_DESIGN = {
    'At rest': {
        'default': 'Platform-managed keys (PMK) — Microsoft manages keys',
        'enhanced': 'Customer-managed keys (CMK) — you control keys in Key Vault',
        'maximum': 'Double encryption (infrastructure + service layer)',
        'decision_guide': [
            {'scenario': 'Most workloads', 'recommendation': 'PMK', 'reason': 'Zero management overhead, meets most compliance'},
            {'scenario': 'Regulated data (HIPAA, PCI)', 'recommendation': 'CMK', 'reason': 'Customer controls key lifecycle, can revoke'},
            {'scenario': 'Highest-assurance / sovereign workloads', 'recommendation': 'CMK in Managed HSM + infrastructure (double) encryption', 'reason': 'Single-tenant FIPS 140-3 Level 3 HSM pool, two independent encryption layers'},
            {'scenario': 'Client-side sensitive fields', 'recommendation': 'Always Encrypted (Azure SQL)', 'reason': 'Data encrypted before reaching server'},
        ],
    },
    'In transit': {
        'minimum': 'TLS 1.2 everywhere (enforce via Azure Policy)',
        'enhanced': 'TLS 1.3 where supported',
        'internal': 'mTLS for service-to-service (e.g., in AKS with service mesh)',
        'vpn': 'IPSec for VPN tunnels (S2S, P2S)',
    },
    'In use': {
        'technology': 'Azure Confidential Computing',
        'how': 'TEE (Trusted Execution Environment) — encrypted memory, even Azure admins cannot access',
        'use_cases': ['Multi-party computation', 'Financial data processing', 'Healthcare data analytics'],
        'azure_services': ['Confidential VMs (DCasv5)', 'Confidential containers (AKS)', 'Always Encrypted with enclaves'],
    },
}

print('=== Encryption Strategy ===\n')

print('--- At Rest ---')
for key, val in ENCRYPTION_DESIGN['At rest'].items():
    if key == 'decision_guide':
        print(f'\n  Decision guide:')
        for d in val:
            print(f'    {d["scenario"]:<35} → {d["recommendation"]:<30} ({d["reason"]})')
    else:
        print(f'  {key}: {val}')

print('\n--- In Transit ---')
for key, val in ENCRYPTION_DESIGN['In transit'].items():
    print(f'  {key}: {val}')

print('\n--- In Use (Confidential Computing) ---')
details = ENCRYPTION_DESIGN['In use']
print(f'  Technology: {details["technology"]}')
print(f'  How: {details["how"]}')
print(f'  Use cases: {", ".join(details["use_cases"])}')
print(f'  Azure services: {", ".join(details["azure_services"])}')

## Hashing vs Encryption vs Tokenization — Know the Difference

A common exam (and interview) trap. Pick the wrong primitive and you either leak data or break functionality.

| Primitive | Reversible? | Use case | Azure feature |
|-----------|-------------|----------|---------------|
| **Hash** (scrypt, Argon2, SHA-256) | ❌ one-way | Password storage, integrity check, dedup | N/A — in code |
| **Symmetric encryption** (AES-GCM) | ✅ with key | Data at rest, data in transit | Storage SSE, Azure SQL TDE, Key Vault |
| **Asymmetric encryption** (RSA, ECC) | ✅ with private key | Key exchange, signing, TLS | Key Vault Keys, certificates |
| **MAC / HMAC** | ❌ (verify only) | Message integrity, webhook auth | APIM policies |
| **Tokenization** | ✅ via lookup table | PCI — replace PAN with token | Purview / 3rd party tokenization services |

Rule of thumb:
- You need to **get the data back** → encryption.
- You only need to **check if it matches** → hashing.
- You need **format-preserving replacement** → tokenization.


In [ ]:
# -------------------------------------------------------------------
# Hashing vs Encryption — runnable demo (stdlib only)
# -------------------------------------------------------------------
import hashlib, hmac, secrets

# --- Hashing: one-way, used for passwords & integrity ---
data = b"patient record #12345"
digest = hashlib.sha256(data).hexdigest()
print("Hash        :", digest[:32], "... (cannot be reversed)")

# --- HMAC: integrity + authenticity with a shared secret ---
key = secrets.token_bytes(32)
tag = hmac.new(key, data, hashlib.sha256).hexdigest()
print("HMAC tag    :", tag[:32], "... (proves sender knew the key)")

# --- "Encryption" concept: reversible with a key ---
# NOTE: real AES-GCM needs the `cryptography` package. For teaching, we use a
# REPEATING-KEY XOR to show only one property: that encryption is reversible with the key.
# This is NOT a one-time pad (the key is reused across the message, which makes it trivially
# breakable) and it provides no integrity protection at all. NEVER use it for real data.
def xor_cipher(msg: bytes, key: bytes) -> bytes:
    full = (key * (len(msg) // len(key) + 1))[:len(msg)]
    return bytes(m ^ k for m, k in zip(msg, full))

ek = secrets.token_bytes(32)
ciphertext = xor_cipher(data, ek)
plaintext  = xor_cipher(ciphertext, ek)
print("Ciphertext  :", ciphertext.hex()[:32], "...")
print("Decrypted   :", plaintext.decode())

print("""
Production crypto (do this, don't roll your own):
  • Python:   cryptography.fernet.Fernet  or  AESGCM
  • .NET:     System.Security.Cryptography.AesGcm
  • Azure:    Storage SSE, Azure SQL TDE, Key Vault Keys (wrap/unwrap)
""")


## Envelope Encryption — How Azure Actually Encrypts Your Data

Every Azure encryption-at-rest feature (Storage SSE, Disk Encryption, SQL TDE with CMK, Cosmos DB CMK) uses the same pattern: **envelope encryption**.

```
                    ┌────────────────────────────────────┐
                    │   Key Encryption Key (KEK)         │
                    │   lives in Azure Key Vault / HSM   │
                    │   you control lifecycle + revoke   │
                    └──────────────┬─────────────────────┘
                                   │ wraps (encrypts)
                                   ▼
                    ┌────────────────────────────────────┐
                    │   Data Encryption Key (DEK)        │
                    │   random per-object / per-blob     │
                    │   stored WITH the ciphertext       │
                    │   (encrypted, never plaintext)     │
                    └──────────────┬─────────────────────┘
                                   │ encrypts
                                   ▼
                    ┌────────────────────────────────────┐
                    │   Your data (blobs, rows, files)   │
                    └────────────────────────────────────┘
```

Why two keys?
- **Fast**: encrypting a 1 TB blob with a symmetric DEK is fast; the KEK only has to wrap one small DEK.
- **Revocable**: revoke the KEK → every DEK is unusable → all data effectively destroyed (crypto-shredding).
- **Compliant**: KEK can live in an HSM; DEK lives with the data. The service never sees the KEK in plaintext.

### Customer-managed key (CMK) flow for Azure Storage

```
1. You create a Key Vault key (KEK).
2. You grant the Storage account identity 'Key Vault Crypto Service Encryption User'.
3. Storage generates a DEK, sends it to Key Vault to be wrapped by your KEK.
4. Storage stores only the wrapped DEK alongside your blobs.
5. On read, Storage asks Key Vault to unwrap the DEK (audited), decrypts the blob.
6. Rotate: point the Storage account at a new KEK version → DEKs get re-wrapped lazily.
7. Revoke: disable the KEK → all Storage reads fail → data is crypto-shredded.
```


In [ ]:
# -------------------------------------------------------------------
# Envelope encryption — conceptual demo
# -------------------------------------------------------------------
# Same *pattern* Azure Storage uses (KEK wraps DEK, DEK encrypts data), in-memory,
# with an insecure toy XOR standing in for AES so we stay stdlib-only.
# The key hierarchy is the lesson here - the cipher is a placeholder.
import secrets, hashlib

# ---- Simulated Key Vault ----
class FakeKeyVault:
    def __init__(self):
        self._keks = {}
    def create_kek(self, name: str) -> None:
        self._keks[name] = secrets.token_bytes(32)
    def wrap(self, kek_name: str, dek: bytes) -> bytes:
        kek = self._keks[kek_name]
        return bytes(a ^ b for a, b in zip(dek, kek))  # toy wrap
    def unwrap(self, kek_name: str, wrapped: bytes) -> bytes:
        kek = self._keks[kek_name]
        return bytes(a ^ b for a, b in zip(wrapped, kek))
    def revoke(self, name: str) -> None:
        del self._keks[name]  # crypto-shredding

# ---- Toy encrypt helper (DO NOT use in production) ----
def toy_encrypt(data: bytes, key: bytes) -> bytes:
    stream = (key * (len(data) // len(key) + 1))[:len(data)]
    return bytes(a ^ b for a, b in zip(data, stream))

kv = FakeKeyVault()
kv.create_kek("adatum-storage-kek")

# ---- Write path ----
patient_record = b"patient=12345 diagnosis=confidential"
dek         = secrets.token_bytes(32)               # 1. fresh DEK per blob
ciphertext  = toy_encrypt(patient_record, dek)      # 2. encrypt data with DEK
wrapped_dek = kv.wrap("adatum-storage-kek", dek)    # 3. wrap DEK with KEK
# Only wrapped_dek + ciphertext are persisted. The plaintext DEK is discarded.
del dek

print("Stored blob     :", ciphertext.hex()[:48], "...")
print("Stored wrapped  :", wrapped_dek.hex()[:48], "...")

# ---- Read path ----
dek_again = kv.unwrap("adatum-storage-kek", wrapped_dek)
plaintext = toy_encrypt(ciphertext, dek_again)
print("Read plaintext  :", plaintext.decode())

# ---- Revocation = crypto-shredding ----
kv.revoke("adatum-storage-kek")
try:
    kv.unwrap("adatum-storage-kek", wrapped_dek)
except KeyError:
    print("After KEK revoke: unwrap fails → data is unreadable (crypto-shredded).")

print("""
Real-world mapping:
  • KEK  = your Key Vault key (HSM-backed for regulated data)
  • DEK  = per-blob / per-database key managed by the service
  • Revoke the KEK in Key Vault → all reads fail → effectively deletes data at scale
""")


In [ ]:
# ===================================================================
# KEY VAULT HIERARCHY DESIGN
# How to structure Key Vaults for an enterprise
# ===================================================================

KEYVAULT_DESIGN = {
    'Hierarchy pattern': {
        'Production Key Vault (per application)': {
            'purpose': 'Application secrets, database connection info, API keys',
            'access': 'Application managed identity (RBAC: Key Vault Secrets User)',
            'network': 'Private endpoint, no public access',
            'features': 'Soft delete + purge protection enabled',
        },
        'Encryption Key Vault (per subscription)': {
            'purpose': 'Customer-managed keys for disk encryption, storage encryption, SQL TDE',
            'access': 'Azure service principals (RBAC: Key Vault Crypto User)',
            'network': 'Private endpoint',
            'features': 'HSM-backed keys for regulated workloads',
        },
        'Certificate Key Vault (shared services)': {
            'purpose': 'TLS certificates for Application Gateway, Front Door, App Services',
            'access': 'Azure services + DevOps pipeline (RBAC: Key Vault Certificates Officer)',
            'network': 'Private endpoint + service-specific access',
            'features': 'Auto-renewal with DigiCert/GlobalSign integration',
        },
        'DevOps Key Vault (per environment)': {
            'purpose': 'CI/CD pipeline secrets (not stored in pipeline variables)',
            'access': 'Pipeline managed identity or workload identity federation',
            'network': 'Allow from Azure DevOps / GitHub Actions IPs',
            'features': 'Separate vaults for dev/staging/prod',
        },
    },
    'Anti-patterns': [
        'Single Key Vault for entire organization (RBAC blast radius too large)',
        'Storing secrets in app settings or environment variables (use Key Vault references)',
        'Using access policies instead of RBAC (RBAC is more granular, Azure recommended)',
        'No soft delete or purge protection (data loss risk)',
        'Public endpoint without IP restrictions (exposure risk)',
    ],
}

print('=== Key Vault Architecture ===\n')
print('--- Recommended hierarchy ---')
for vault, details in KEYVAULT_DESIGN['Hierarchy pattern'].items():
    print(f'\n  {vault}:')
    for key, val in details.items():
        print(f'    {key}: {val}')

print('\n\n--- Anti-patterns (avoid these) ---')
for ap in KEYVAULT_DESIGN['Anti-patterns']:
    print(f'  ✗ {ap}')

## Database Security Architecture

### Securing Azure data services — defense in depth:

| Service | Network | Auth | Encryption | Threat Detection | Audit |
|---------|---------|------|------------|-----------------|-------|
| **Azure SQL** | Private Endpoint | Microsoft Entra-only authentication (SQL auth disabled) | TDE (PMK or CMK) + Always Encrypted for PII | Defender for SQL | SQL Audit → Sentinel |
| **Cosmos DB** | Private Endpoint | Entra ID RBAC (disable keys) | Encrypted at rest (PMK/CMK) | Defender for Cosmos DB | Diagnostic logs |
| **Synapse** | Managed VNet + Private Endpoints | Entra ID + managed identity | TDE + column-level encryption | Defender for SQL | Synapse audit logs |
| **Storage** | Private Endpoint + firewall | Entra ID RBAC (disable shared key) | SSE (PMK/CMK) + client-side | Defender for Storage | Storage analytics logs |
| **Azure Database for PostgreSQL** | Private Endpoint | Entra ID + password | Encrypted at rest (PMK/CMK) | Defender for databases | pgaudit + diagnostic logs |

## Azure SQL — Defense-in-Depth Layers Applied

A single Azure SQL database should use **multiple** overlapping controls. Assume each layer will fail; the next one should catch it.

```
        ┌─────────────────────────────────────────────────────────────┐
        │  1. NETWORK     Private Endpoint + deny public access       │
        │                  → attacker on the internet can't even      │
        │                    see the server                            │
        ├─────────────────────────────────────────────────────────────┤
        │  2. AUTH        Entra ID only (SQL auth disabled)           │
        │                 Managed identity from app, MFA for humans   │
        │                  → stolen password alone is not enough      │
        ├─────────────────────────────────────────────────────────────┤
        │  3. AUTHZ       RBAC + row-level security (RLS)             │
        │                  → app can only see tenants it owns         │
        ├─────────────────────────────────────────────────────────────┤
        │  4. DATA        Always Encrypted for PII columns            │
        │                 TDE with CMK for the rest                   │
        │                  → DBA sees ciphertext, not plaintext       │
        ├─────────────────────────────────────────────────────────────┤
        │  5. MASKING     Dynamic Data Masking for dev/support users  │
        │                  → call-center sees ****1234 not full SSN   │
        ├─────────────────────────────────────────────────────────────┤
        │  6. DETECTION   Defender for SQL                            │
        │                  → alerts on SQLi, brute force, exfil       │
        ├─────────────────────────────────────────────────────────────┤
        │  7. AUDIT       SQL Audit → Log Analytics → Sentinel        │
        │                  → every query retained 7 years for HIPAA   │
        ├─────────────────────────────────────────────────────────────┤
        │  8. BACKUP      Geo-redundant backups + LTR                 │
        │                  → ransomware / accidental drop recovery    │
        └─────────────────────────────────────────────────────────────┘
```

Each layer maps to an exam-relevant Azure feature. Memorize the stack.


In [ ]:
# ===================================================================
# DEFENDER FOR DATA SERVICES ARCHITECTURE
# ===================================================================

DEFENDER_DATA_SERVICES = [
    {
        'service': 'Defender for Storage',
        'detections': ['Malware upload detection', 'Anomalous access patterns', 'Data exfiltration',
                       'Anonymous access from suspicious IP', 'Unusual amount of data extracted'],
        'unique_feature': 'Malware Scanning — scans blobs on upload using Microsoft Antimalware engine',
        'design_tip': 'Enable per-storage account. Set up blob index tags for malware scan results.',
    },
    {
        'service': 'Defender for SQL',
        'detections': ['SQL injection', 'Brute force', 'Anomalous query patterns',
                       'Data exfiltration alerts', 'Potential vulnerability (missing patches)'],
        'unique_feature': 'Vulnerability Assessment — scans for misconfigurations and recommends fixes',
        'design_tip': 'Enable at subscription level for all Azure SQL. Configure email alerts to DBA + SOC.',
    },
    {
        'service': 'Defender for Cosmos DB',
        'detections': ['Suspicious access from Tor', 'Key extraction attempt',
                       'Anomalous query volume', 'Unusual data export'],
        'unique_feature': 'Detects attacks on Cosmos DB API layer (SQL, MongoDB, Cassandra APIs)',
        'design_tip': 'Enable for production Cosmos DB accounts. Disable key-based access in favor of RBAC.',
    },
    {
        'service': 'Defender for Key Vault',
        'detections': ['Unusual key vault access', 'Access from suspicious IP/Tor',
                       'High volume of operations', 'Policy change'],
        'unique_feature': 'Detects access pattern anomalies specific to secrets/keys/certificates',
        'design_tip': 'Low cost, high value. Enable on all Key Vaults; the plan moved to a fixed pricing model in 2026. Alert on any Tor/anonymous access.',
    },
]

print('=== Defender for Data Services ===\n')
for svc in DEFENDER_DATA_SERVICES:
    print(f'\n--- {svc["service"]} ---')
    print(f'  Detections: {", ".join(svc["detections"][:3])}...')
    print(f'  Unique: {svc["unique_feature"]}')
    print(f'  Design tip: {svc["design_tip"]}')

## Microsoft 365 Security Architecture

### M365 security: Purview + Defender for Office 365 + Intune

```
┌──────────────────────────────────────────────────────────────────────────┐
│                     M365 SECURITY ARCHITECTURE                          │
│                                                                          │
│  ┌────────────────────────┐  ┌───────────────────────────────────────┐  │
│  │  DEFENDER FOR OFFICE   │  │  MICROSOFT PURVIEW                   │  │
│  │  365 (Plan 2)          │  │                                       │  │
│  │                        │  │  Data lifecycle:                      │  │
│  │  • Safe Attachments    │  │  • Sensitivity labels                 │  │
│  │  • Safe Links          │  │  • DLP (Exchange, SharePoint, Teams)  │  │
│  │  • Anti-phishing       │  │  • Retention policies                 │  │
│  │  • Attack simulation   │  │  • Records management                 │  │
│  │  • Automated invest.   │  │  • eDiscovery (Premium)               │  │
│  │  • Threat Explorer     │  │  • Insider Risk Management            │  │
│  │  • Real-time detections│  │  • Communication Compliance           │  │
│  └────────────────────────┘  │  • Information Barriers               │  │
│                               └───────────────────────────────────────┘  │
│                                                                          │
│  ┌────────────────────────────────────────────────────────────────────┐  │
│  │  MICROSOFT INTUNE                                                  │  │
│  │                                                                    │  │
│  │  • Device compliance policies (required for CA)                    │  │
│  │  • App protection policies (MAM for BYOD)                          │  │
│  │  • Configuration profiles (security baselines)                     │  │
│  │  • Windows Autopilot (zero-touch provisioning)                     │  │
│  │  • Endpoint Privilege Management (EPM)                             │  │
│  │  • Remote actions (wipe, lock, retire)                             │  │
│  └────────────────────────────────────────────────────────────────────┘  │
└──────────────────────────────────────────────────────────────────────────┘
```

In [ ]:
# ===================================================================
# MICROSOFT 365 COPILOT — DATA SECURITY DESIGN
# Key topic: securing data BEFORE deploying Copilot
# ===================================================================

COPILOT_SECURITY_CHECKLIST = [
    {
        'phase': 'Pre-deployment (Critical)',
        'actions': [
            {'action': 'Review SharePoint/OneDrive permissions', 'why': 'Copilot can access anything the user can. Overshared sites = data exposure.',
             'tool': 'SharePoint Advanced Management + Purview Data Access Governance'},
            {'action': 'Deploy sensitivity labels to high-value content', 'why': 'Labels travel with content. Copilot respects label restrictions.',
             'tool': 'Purview Information Protection with auto-labeling'},
            {'action': 'Remove "Everyone except external users" permissions', 'why': 'This default permission means Copilot shows content to everyone.',
             'tool': 'SharePoint site access review + sensitivity labels'},
            {'action': 'Implement DLP policies', 'why': 'Prevent Copilot-generated content from containing/sharing sensitive data.',
             'tool': 'Purview DLP for Teams, SharePoint, Exchange'},
        ],
    },
    {
        'phase': 'Deployment',
        'actions': [
            {'action': 'Assign Copilot licenses to specific groups (not all users)', 'why': 'Staged rollout limits exposure during initial deployment.',
             'tool': 'Entra ID group-based licensing'},
            {'action': 'Configure Copilot audit logging', 'why': 'Track what users ask Copilot and what data it accesses.',
             'tool': 'Purview Audit (Standard captures Copilot interactions; Premium adds longer retention and high-value events)'},
            {'action': 'Enable Adaptive Protection', 'why': 'Automatically restrict Copilot for high-risk users.',
             'tool': 'Purview Insider Risk + Conditional Access integration'},
        ],
    },
    {
        'phase': 'Post-deployment monitoring',
        'actions': [
            {'action': 'Monitor Copilot usage patterns', 'why': 'Detect anomalous usage (bulk queries, sensitive topic probing).',
             'tool': 'Purview Audit + Defender for Cloud Apps'},
            {'action': 'Review Copilot-generated content for data leakage', 'why': 'Ensure Copilot isn\'t surfacing confidential data inappropriately.',
             'tool': 'DLP policy alerts + content explorer'},
            {'action': 'Regular access reviews for Copilot-licensed users', 'why': 'Ensure only appropriate users retain Copilot access.',
             'tool': 'Entra ID Access Reviews'},
        ],
    },
]

print('=== Microsoft 365 Copilot Data Security Checklist ===\n')
for phase_data in COPILOT_SECURITY_CHECKLIST:
    print(f'\n{"=" * 70}')
    print(f'{phase_data["phase"]}')
    print(f'{"=" * 70}')
    for a in phase_data['actions']:
        print(f'\n  Action: {a["action"]}')
        print(f'  Why: {a["why"]}')
        print(f'  Tool: {a["tool"]}')

In [ ]:
# ===================================================================
# INTUNE DEVICE MANAGEMENT DESIGN
# ===================================================================

INTUNE_DESIGN = {
    'Corporate-owned devices': {
        'enrollment': 'Windows Autopilot (zero-touch) / Apple DEP / Android Enterprise fully managed',
        'management': 'Full MDM — Intune controls entire device',
        'security': [
            'Device compliance policy (BitLocker, firewall, antivirus, OS version)',
            'Security baseline (Microsoft recommended settings)',
            'Endpoint Privilege Management (EPM) — elevate apps, not users',
            'Windows LAPS for local admin passwords',
            'Conditional Access: require device compliance for M365 access',
        ],
    },
    'BYOD (personal devices)': {
        'enrollment': 'MAM-only (no device enrollment required)',
        'management': 'App-level protection — Intune manages only corporate apps/data',
        'security': [
            'App protection policy (prevent copy/paste to personal apps)',
            'Selective wipe (remove only corporate data, keep personal)',
            'Require PIN/biometrics for corporate apps',
            'Prevent screenshots of corporate content',
            'Conditional Access: require app protection policy',
        ],
    },
    'Kiosks and shared devices': {
        'enrollment': 'Windows Autopilot self-deploying / Shared device mode',
        'management': 'Locked-down configuration, shared device mode (Android/iOS)',
        'security': [
            'Kiosk mode (single or multi-app)',
            'Shared device mode with automatic sign-out',
            'No user data persistence between sessions',
            'Defender for Endpoint in passive mode',
        ],
    },
}

print('=== Intune Device Management Architecture ===\n')
for device_type, details in INTUNE_DESIGN.items():
    print(f'\n--- {device_type} ---')
    print(f'  Enrollment: {details["enrollment"]}')
    print(f'  Management: {details["management"]}')
    print(f'  Security controls:')
    for control in details['security']:
        print(f'    • {control}')

---

## CAPSTONE SCENARIO: Design a Complete Security Architecture

This is your final exercise. You'll design a comprehensive security architecture for a fictional company, using everything you've learned across all SC-100 labs.

In [ ]:
# ===================================================================
# CAPSTONE: Design complete security architecture for Adatum Corp
# ===================================================================

CAPSTONE_SCENARIO = """
═══════════════════════════════════════════════════════════════════════
                    CAPSTONE SCENARIO: ADATUM CORP
═══════════════════════════════════════════════════════════════════════

COMPANY: Adatum Corporation
INDUSTRY: Healthcare technology (SaaS platform for hospitals)
EMPLOYEES: 2,500
CUSTOMERS: 200 hospitals using Adatum's cloud platform
REGULATIONS: HIPAA, HITRUST, SOC 2 Type II

CURRENT INFRASTRUCTURE:
  Azure:
    - 3 Entra ID tenants (main + 2 acquisitions)
    - 10 subscriptions under 1 management group
    - AKS clusters running the SaaS platform (customer-facing)
    - Azure SQL with patient data (PHI) from 200 hospitals
    - Cosmos DB for real-time health monitoring data
    - Azure OpenAI for clinical decision support (new product)
    - Storage accounts for medical imaging (DICOM files)
    - Azure Functions for data processing pipelines
  
  AWS:
    - Legacy analytics platform (50 EC2 instances)
    - S3 buckets with historical patient data
  
  M365:
    - E5 licenses for all employees
    - Teams used for customer support channels
    - SharePoint with product documentation and SOC 2 evidence
  
  On-premises:
    - Development lab with test servers
    - Cisco ASA firewalls (legacy)

SECURITY TEAM:
  - CISO + 3 security engineers
  - No dedicated SOC (security engineers handle incidents part-time)
  - Using Splunk (expensive, contract ending in 6 months)

KNOWN ISSUES:
  1. No formal threat model for the SaaS platform
  2. 20 Global Admins across 3 tenants
  3. Service principal secrets in GitHub repos (discovered during audit)
  4. No data classification — all patient data treated the same
  5. Azure SQL has public endpoints in dev/staging
  6. Splunk costs $500K/year, limited to Azure logs only
  7. No endpoint protection on developer workstations
  8. AWS accounts not monitored for security
  9. Microsoft 365 Copilot deployment planned (board initiative)
  10. Azure OpenAI using API keys (not managed identity)

BOARD PRIORITIES:
  - Pass HITRUST certification within 12 months
  - Deploy Microsoft 365 Copilot safely
  - Reduce security tool spend (Splunk replacement)
  - "Zero Trust" architecture (board read an article about it)
"""

print(CAPSTONE_SCENARIO)

In [ ]:
# ===================================================================
# CAPSTONE SOLUTION: Security architecture design for Adatum
# ===================================================================

ARCHITECTURE = {
    '1. SECURITY OPERATIONS (replace Splunk, build SOC capability)': {
        'design': [
            'Deploy Sentinel + Defender XDR (unified) — replaces Splunk at lower cost',
            'Sentinel workspace in primary tenant, Azure Lighthouse for cross-tenant',
            'Defender XDR: included with E5 — no extra cost for M365 threat detection',
            'AWS connector for CloudTrail + GuardDuty integration',
            'SOAR: automate 80% of Tier-1 alerts (phishing, brute force, impossible travel)',
            'MITRE ATT&CK coverage targeting: Initial Access, Credential Access, Exfiltration',
        ],
        'cost_saving': 'Splunk $500K → Sentinel ~$150K (E5 data included free)',
    },
    '2. IDENTITY & ACCESS (Zero Trust foundation)': {
        'design': [
            'Consolidate 3 tenants → 1 primary (long-term), cross-tenant access policies (short-term)',
            'Reduce Global Admins: 20 → 3 permanent (+ 2 break-glass)',
            'PIM for all admin roles, phishing-resistant MFA (FIDO2) for control plane',
            'Conditional Access: 8-policy baseline (MFA, device compliance, risk-based)',
            'Workload identity migration: GitHub secrets → workload identity federation',
            'Azure OpenAI: migrate from API keys to managed identity',
        ],
        'quick_win': 'Break-glass accounts + PIM + phishing-resistant MFA = biggest risk reduction',
    },
    '3. INFRASTRUCTURE SECURITY (posture + endpoints)': {
        'design': [
            'Defender CSPM (paid) for attack path analysis across Azure + AWS',
            'Defender plans: Servers P2 (prod), Containers, SQL, Storage, Key Vault, Cosmos DB',
            'Azure Arc for AWS EC2 instances (50 servers)',
            'Azure Policy: deny public endpoints on SQL/Storage/Cosmos (all environments)',
            'Defender for Endpoint P2 on all developer workstations via Intune',
            'Network: Private Endpoints for all PaaS, Azure Firewall in hub',
        ],
        'quick_win': 'Azure Policy deny public SQL endpoints — prevents issue #5 immediately',
    },
    '4. APPLICATION SECURITY (SaaS platform)': {
        'design': [
            'Threat model the SaaS platform using STRIDE (address issue #1)',
            'DevSecOps: GitHub Advanced Security (CodeQL + secret scanning) for all repos',
            'AKS: private cluster, workload identity, Defender for Containers',
            'Azure OpenAI: private endpoint, managed identity, content filtering, audit logging',
            'WAF: Azure Front Door with OWASP rule set for customer-facing APIs',
            'API security: APIM with OAuth 2.0, rate limiting, IP restrictions per hospital',
        ],
        'quick_win': 'Secret scanning catches leaked credentials in GitHub immediately',
    },
    '5. DATA SECURITY (HIPAA compliance + Copilot readiness)': {
        'design': [
            'Sensitivity labels: PHI, Confidential, General, Public',
            'Auto-labeling for PHI data (SITs for medical record numbers, SSN, diagnosis codes)',
            'DLP: prevent PHI sharing via email, Teams, SharePoint external sharing',
            'Azure SQL: Always Encrypted for PHI columns, CMK encryption, Defender for SQL',
            'Microsoft 365 Copilot pre-deployment: SharePoint permission audit, sensitivity labels, DLP',
            'Medical imaging (Storage): Defender for Storage malware scanning, immutable storage',
        ],
        'quick_win': 'Data classification is prerequisite for HITRUST — start immediately',
    },
    '6. COMPLIANCE (HITRUST certification path)': {
        'design': [
            'Defender for Cloud: HITRUST regulatory compliance initiative',
            'Azure Policy: HITRUST built-in initiative at management group level',
            'Purview Compliance Manager: track HITRUST control implementation',
            'Audit logging: all services → Sentinel (7-year retention for HIPAA)',
            'Access reviews: quarterly for all admin and privileged roles',
            'Insider Risk Management: departing employee policy (SOC 2 requirement)',
        ],
        'quick_win': 'Assign HITRUST policy initiative to see current compliance score',
    },
}

print('═' * 80)
print('      ADATUM CORP — COMPLETE SECURITY ARCHITECTURE DESIGN')
print('═' * 80)

for domain, details in ARCHITECTURE.items():
    print(f'\n{"─" * 80}')
    print(f'{domain}')
    print(f'{"─" * 80}')
    for item in details['design']:
        print(f'  • {item}')
    for key in ['cost_saving', 'quick_win']:
        if key in details:
            label = 'Cost saving' if key == 'cost_saving' else 'Quick win'
            print(f'  >>> {label}: {details[key]}')

print(f'\n{"═" * 80}')
print('IMPLEMENTATION PRIORITY ORDER:')
print('  Phase 1 (0-30 days):  Identity hardening + break-glass + Azure Policy deny public endpoints')
print('  Phase 2 (30-90 days): Sentinel deployment (replace Splunk) + Defender plans + DevSecOps')
print('  Phase 3 (90-180 days): Data classification + Microsoft 365 Copilot deployment + threat modeling')
print('  Phase 4 (180-365 days): HITRUST certification + tenant consolidation + full Zero Trust')
print(f'{"═" * 80}')

In [ ]:
# ===================================================================
# FINAL QUIZ: Data Security Architecture
# ===================================================================

QUIZ = [
    {
        'question': 'Adatum stores PHI (patient health information) in Azure SQL. Which encryption\n'
                    'approach ensures data is encrypted EVEN from database administrators?',
        'options': {
            'A': 'Transparent Data Encryption (TDE) with customer-managed keys',
            'B': 'Always Encrypted with client-side key management',
            'C': 'Azure Disk Encryption for the SQL managed instance',
            'D': 'TLS 1.3 for all database connections',
        },
        'answer': 'B',
        'explanation': 'Always Encrypted encrypts data BEFORE it reaches SQL Server — the database engine '
                       'never sees plaintext. DBAs with full SQL permissions cannot read the encrypted columns. '
                       'TDE encrypts at-rest (disk level) but data is plaintext in memory and visible to DBAs. '
                       'TLS is for transit, not at-rest protection.',
    },
    {
        'question': 'Before deploying Microsoft 365 Copilot, what is the MOST critical preparatory step?',
        'options': {
            'A': 'Enable Defender for Office 365 Plan 2',
            'B': 'Review and fix SharePoint/OneDrive oversharing and permissions',
            'C': 'Deploy DLP policies for all M365 workloads',
            'D': 'Configure audit logging for Copilot interactions',
        },
        'answer': 'B',
        'explanation': 'Copilot accesses data the user has permission to access. If SharePoint sites are '
                       'overshared (e.g., "Everyone except external users" default), Copilot will surface '
                       'that data to all employees. Fixing permissions BEFORE deployment prevents data exposure. '
                       'DLP and audit are important but secondary to fixing the root access problem.',
    },
    {
        'question': 'Adatum needs to detect if a departing employee downloads large amounts of patient\n'
                    'data before their last day. Which tool is BEST for this?',
        'options': {
            'A': 'Purview DLP policies',
            'B': 'Defender for Cloud Apps anomaly detection',
            'C': 'Purview Insider Risk Management with HR connector',
            'D': 'Microsoft Sentinel analytics rule',
        },
        'answer': 'C',
        'explanation': 'Insider Risk Management specifically has a "Data theft by departing users" policy '
                       'template. It correlates HR signals (resignation date from HR connector) with data '
                       'activity (download volume, email forwarding, USB copy). DLP prevents sharing but '
                       'doesn\'t correlate with HR events. Defender for Cloud Apps detects anomalies but '
                       'lacks the HR context.',
    },
    {
        'question': 'Adatum is replacing Splunk ($500K/year) with Microsoft security tools.\n'
                    'Which approach provides the BEST cost optimization?',
        'options': {
            'A': 'Microsoft Sentinel with every table on the Analytics plan',
            'B': 'Microsoft Sentinel with tiered table plans: Analytics + Basic + Auxiliary, plus long-term retention',
            'C': 'Defender XDR only (no Sentinel)',
            'D': 'Microsoft Sentinel with Commitment Tier pricing',
        },
        'answer': 'B',
        'explanation': 'Table plans dramatically reduce Sentinel cost: security-critical data on the ANALYTICS '
                       'plan (full KQL, alerting), medium-touch logs on BASIC (cheap ingestion, single-table '
                       'queries), verbose/audit data on AUXILIARY (cheapest), then LONG-TERM RETENTION - the '
                       'setting formerly called the "Archive tier" - for compliance, searched with search jobs. '
                       'Defender XDR alerts and incidents ingest free, but Entra ID sign-in logs are billable, '
                       'so "E5 makes Sentinel free" is a trap. Commitment tiers help too, but plan selection '
                       'has the biggest impact.',
    },
]

print('=== Data Security Architecture — Final Quiz ===\n')
for i, q in enumerate(QUIZ, 1):
    print(f'Question {i}:')
    print(f'{q["question"]}\n')
    for key, option in q['options'].items():
        marker = '>>>' if key == q['answer'] else '   '
        print(f'  {marker} {key}. {option}')
    print(f'\n  Answer: {q["answer"]}')
    print(f'  Why: {q["explanation"]}')
    print()

print('\n' + '═' * 80)
print('Congratulations! You\'ve completed all SC-100 labs.')
print('You\'ve designed security architectures spanning:')
print('  • Security best practices & frameworks (Lab 1)')
print('  • Security operations, identity & compliance (Lab 2)')
print('  • Infrastructure security (Lab 3)')
print('  • Application and data security (Lab 4)')
print('═' * 80)